In [1]:
import pandas as pd
import numpy as np
import os
import math
import seaborn as sns
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import GofChisquarePower
from sklearn.cluster import KMeans
import sklearn.metrics as sil
from ratio import find_power_vals

In [2]:
humans = pd.read_pickle('./data/phenotype/humans_maf_popstrat.pkl')

In [3]:
humans

,sample,sample.1,pop,super_pop,gender,pheno_maf_0.01,pheno_maf_0.016,pheno_maf_0.023,pheno_maf_0.035,pheno_maf_0.049,pheno_maf_0.067,pheno_maf_0.091,pheno_maf_0.12,pheno_maf_0.16,pheno_maf_0.21,pheno_maf_0.26,pheno_maf_0.32,pheno_maf_0.38
0,HG00096,HG00096,GBR,EUR,male,-0.374728,-0.055512,-0.789928,3.361361,-0.859756,0.058922,0.737843,1.370635,1.012293,-0.078382,0.403039,-1.359728,-3.057570
1,HG00097,HG00097,GBR,EUR,female,-0.443200,-0.068150,-0.704813,0.034578,-0.873216,0.163947,0.835553,0.306773,-0.448527,-0.404108,-0.188747,-0.190882,-0.253866
2,HG00099,HG00099,GBR,EUR,female,-0.770721,-0.050913,-0.946380,-0.073258,-0.906715,0.720611,0.282926,0.404328,0.465232,-0.861369,-0.607776,-0.602197,-0.050176
3,HG00100,HG00100,GBR,EUR,female,-0.431489,0.038862,-0.590873,0.061772,-0.928455,1.127265,1.994694,0.020818,-0.965322,-0.898903,1.063723,-1.017250,-2.174640
4,HG00101,HG00101,GBR,EUR,male,-0.450789,-0.019231,-0.564662,-0.001784,-0.971067,0.080965,0.321142,0.251472,-0.020584,-0.768223,-0.846141,0.145603,-1.981969
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2499,NA21137,NA21137,GIH,SAS,female,-0.037137,-0.020363,-0.870381,1.635897,-0.931037,0.608615,-0.266781,-0.541917,0.426657,-0.268584,-1.300425,0.351405,1.046404
2500,NA21141,NA21141,GIH,SAS,female,-0.575891,-2.907584,-0.151323,-0.040280,-0.935228,0.549886,-0.318017,-0.558477,-0.037681,-0.602524,-0.684395,1.412411,-0.984465
2501,NA21142,NA21142,GIH,SAS,female,-0.493638,-0.070322,0.235336,0.109402,-0.893528,2.940449,-0.159810,-0.773118,-0.751767,-0.086756,-1.742997,0.764024,-0.009483
2502,NA21143,NA21143,GIH,SAS,female,-0.484568,-0.303935,1.049456,0.025503,-0.969741,2.635246,-0.682898,-0.830271,-0.121528,0.283709,0.503010,0.236412,1.411802


In [4]:
wanted_maf = '0.1_0.1'



In [5]:
snps = [f for f in os.listdir('./data/genotype') if f.split("_maf_")[1].split('.pkl')[0] == wanted_maf]
for snp in snps:
    maf = snp.split("_maf_")[1].split('.pkl')[0]
    print(maf)
    path_snp = f"./data/genotype/{snp}"
    print(path_snp)
    geno = pd.read_pickle(path_snp)

0.1_0.1
./data/genotype/60_2_120_60_maf_0.1_0.1.pkl


In [6]:
snp_num = 0
geno[geno.columns[snp_num]].value_counts()
num_maj = geno[geno.columns[snp_num]].value_counts()[1]
num_het = geno[geno.columns[snp_num]].value_counts()[0]
try:
    num_min = geno[geno.columns[snp_num]].value_counts()[-1]
except Exception as e:
    num_min = 0
total_humans = num_maj + num_het + num_min
maf = (num_min*2 + num_het)/(total_humans*2)
maf

0.10263578274760383

In [16]:
min_power = 80
[a, b, c] = find_power_vals(min_power)
ratio = a*maf**2 + b*maf + c

true_mean = 0
true_effect = 5
true_sd = true_effect*ratio

mean_min = true_mean
mean_het = true_mean
mean_maj = true_mean
sigma_min = true_sd
sigma_het = true_sd
sigma_maj = true_sd

iteration = 2000

In [18]:

pheno = []
for var in list(geno[geno.columns[snp_num]]):
    if var == 1:
        pheno.append(np.random.normal(mean_maj, sigma_maj, 1)[0])
    elif var == 0:
        pheno.append(np.random.normal(mean_het, sigma_het, 1)[0])
    else:
        pheno.append(np.random.normal(mean_min, sigma_min, 1)[0])
df_pheno = pd.DataFrame(data = {"geno": list(geno[geno.columns[snp_num]]), "pheno_null": pheno})

In [19]:
df_pheno

,geno,pheno_null
0,1,-32.838115
1,1,71.254342
2,1,-19.894795
3,1,11.255956
4,1,7.148350
...,...,...
2499,0,38.854941
2500,1,22.827776
2501,0,-71.822900
2502,1,-42.983359


In [20]:
maf_pop = 0.01
pheno_pop = humans[f"pheno_maf_{maf_pop}"]

In [21]:
df_pheno[f"pheno_strat_{maf_pop}"] = pheno_pop + df_pheno['pheno_null']

In [22]:
df_pheno

,geno,pheno_null,pheno_strat_0.01
0,1,-32.838115,-33.212843
1,1,71.254342,70.811142
2,1,-19.894795,-20.665516
3,1,11.255956,10.824468
4,1,7.148350,6.697561
...,...,...,...
2499,0,38.854941,38.817804
2500,1,22.827776,22.251885
2501,0,-71.822900,-72.316537
2502,1,-42.983359,-43.467927
